In [ ]:
## Gravitational Waves and Machine Learning
# Author: Nathan Steinle, Feb. 2026

# =============================================================
# This notebook is designed for a lecture on waves, GW detection theory, and
# a simple application of machine learning classification. 

# Takes about 3 minutes to run. 

# =============================================================
# Run this in a virtual python environment. Create one with the following commands (replace the path in MYENV):
#         MYVENV=<home path>/waves_env
#         virtualenv ${MYVENV}
#         source ${MYVENV}/bin/activate
#         pip3 install numpy scipy matplotlib pathos astropy pycbc scikit-learn jupyter jupyterlab




# =============================================================
# PART 0: Imports and Setup
# =============================================================

import numpy as np
import matplotlib.pyplot as plt

from pycbc.catalog import Merger
from pycbc.types import TimeSeries
from pycbc.filter import matched_filter
from pycbc.psd import interpolate, inverse_spectrum_truncation
from pycbc.waveform import get_td_waveform

np.random.seed(1234)








# =============================================================
# PART 1: Classical Matched Filtering (GWOSC LOSC tutorial)
# =============================================================

# -----------------------------
# 1. Load real LIGO data
# -----------------------------

m = Merger('GW150914')
strain = m.strain('H1')

# -----------------------------
# 2. Conditioning 
# -----------------------------

strain = strain.crop(2, 2)
strain = strain.highpass_fir(30, 512)
strain = strain.resample(1.0 / 2048)

# -----------------------------
# 3. PSD estimation 
# -----------------------------

psd = strain.psd(4)
psd = interpolate(psd, strain.delta_f)
psd = inverse_spectrum_truncation(
    psd,
    int(4 * strain.sample_rate),
    low_frequency_cutoff=30
)

# -----------------------------
# 4. Template waveform (LOSC tutorial)
# -----------------------------

template, _ = get_td_waveform(
    approximant='TaylorF2',
    mass1=36,
    mass2=29,
    delta_t=strain.delta_t,
    f_lower=30
)

# --- Match template length to data length (required for FFT)
if len(template) > len(strain):
    template = template.crop(0, template.duration - strain.duration)
elif len(template) < len(strain):
    pad = len(strain) - len(template)
    template = TimeSeries(
        np.pad(template.numpy(), (0, pad)),
        delta_t=template.delta_t,
        epoch=template.start_time
    )

assert len(template) == len(strain)

# -----------------------------
# 5. Matched filter to compute the SNR time series (LOSC tutorial)
# -----------------------------

snr_ts = matched_filter(
    template,
    strain,
    psd=psd,
    low_frequency_cutoff=30
)

snr_ts = snr_ts.crop(4, 4)

snr = np.abs(snr_ts.numpy())
times = snr_ts.sample_times


plt.figure(figsize=(7,6))
plt.plot(times, snr)
plt.xlabel("t")
plt.ylabel("SNR")
plt.title("BBH Signal-to-noise ratio vs time")
# plt.legend()
# plt.grid(True)
plt.tight_layout()
plt.show()









In [ ]:


# =============================================================
# PART 2: Machine Learning Classification of BBH vs BNS Waveforms
# =============================================================

# In this section we abandon "detection" and instead pose a clean,
# well-defined ML task:
#
#   Given a theoretical inspiral–merger waveform, can a classifier
#   distinguish a binary black hole (BBH) from a binary neutron star (BNS)?
#
# This is a physically meaningful task because:
#   • BBHs are more massive → lower merger frequency, higher amplitude
#   • BNSs are lighter     → longer signals, higher merger frequency
#
# The waveforms are fully theoretical (PyCBC), noiseless, and aligned
# to a common time window. This isolates *morphology*, not detection.

# =============================================================
# PART 2A: Generate a waveform bank (theory only)
# =============================================================

from pycbc.waveform import get_td_waveform
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

fs = 4096
delta_t = 1.0 / fs
f_lower = 30

n_bns = 200
n_bbh = 200
window_size = 4096   # 1 second window

X = []
y = []

# -----------------------------
# BNS waveforms
# -----------------------------

for _ in range(n_bns):
    m1 = np.random.uniform(1.2, 1.6)
    m2 = np.random.uniform(1.2, 1.6)

    hp, _ = get_td_waveform(
        approximant="TaylorF2",
        mass1=m1,
        mass2=m2,
        delta_t=delta_t,
        f_lower=f_lower
    )

    h = hp.numpy()
    h = h[-window_size:]

    if len(h) < window_size:
        h = np.pad(h, (window_size - len(h), 0))

    X.append(h)
    y.append(0)   # BNS label

# -----------------------------
# BBH waveforms
# -----------------------------

for _ in range(n_bbh):
    m1 = np.random.uniform(8, 30)
    m2 = np.random.uniform(8, 30)

    hp, _ = get_td_waveform(
        approximant="IMRPhenomD",
        mass1=m1,
        mass2=m2,
        delta_t=delta_t,
        f_lower=f_lower
    )

    h = hp.numpy()
    h = h[-window_size:]

    if len(h) < window_size:
        h = np.pad(h, (window_size - len(h), 0))

    X.append(h)
    y.append(1)   # BBH label

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

# =============================================================
# PART 2B: Feature scaling and dimensionality reduction
#          Train / test split (RAW data first)
# =============================================================

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Log-amplitude scaling (dynamic range stabilization)
X_train_log = np.log10(np.abs(X_train_raw) + 1e-22)
X_test_log  = np.log10(np.abs(X_test_raw)  + 1e-22)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_log)
X_test_scaled  = scaler.transform(X_test_log)

# PCA captures dominant waveform morphology
pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print("Explained variance (first 5 PCs):",
      pca.explained_variance_ratio_[:5])




# =============================================================
# PART 2C: Supervised classification
# =============================================================

clf = LogisticRegression(max_iter=500)
clf.fit(X_train_pca, y_train)

y_pred = clf.predict(X_test_pca)

print(classification_report(
    y_test, y_pred,
    target_names=["BNS", "BBH"]
))




# =============================================================
# PART 2D: Visualization in PCA space (fixed for train/test split)
# =============================================================

# Combine train and test PCA data for a full view
X_full_pca = np.vstack([X_train_pca, X_test_pca])
y_full = np.hstack([y_train, y_test])

plt.figure(figsize=(7,6))
plt.scatter(
    X_full_pca[y_full==0, 0], X_full_pca[y_full==0, 1],
    label="BNS", alpha=0.7
)
plt.scatter(
    X_full_pca[y_full==1, 0], X_full_pca[y_full==1, 1],
    label="BBH", alpha=0.7
)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("BBH vs BNS Waveforms (Time Domain, PCA)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()





# =============================================================
# PART 2E: Interpretation
# =============================================================
# The classifier succeeds because BBH and BNS waveforms differ
# systematically in:
#   • duration
#   • amplitude scale
#   • chirp rate (frequency evolution)
#
# PCA shows that only a few components capture most of the
# physically relevant variance. The ML model is therefore
# learning *mass-dependent waveform morphology*, not noise.
#
# This makes for a clean example:
#   classical physics → waveform structure → ML classification


In [ ]:
# =============================================================
# Visualization: overlay true vs predicted labels (one column)
# =============================================================

n_plot = 10
idx = np.random.choice(len(X_test_raw), n_plot, replace=False)

time = np.arange(window_size) * delta_t

fig, axes = plt.subplots(n_plot, 1, figsize=(10, 2.0 * n_plot), sharex=True)

for i, k in enumerate(idx):
    label_true = "BBH" if y_test[k] == 1 else "BNS"
    label_pred = "BBH" if y_pred[k] == 1 else "BNS"

    # Plot true waveform
    axes[i].plot(time, X_test_raw[k], color='blue', lw=1.5, label=f"True: {label_true}")

    # Overlay predicted waveform (same strain, but line style shows label)
    axes[i].plot(time, X_test_raw[k],
                 color='red',
                 lw=1.2,
                 linestyle='--',
                 label=f"Predicted: {label_pred}")

    axes[i].set_ylabel("Strain")
    axes[i].legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel("Time (s)")

plt.suptitle("BBH vs BNS Waveforms: True vs Predicted Labels (Overlay)", y=0.92)
plt.tight_layout()
plt.show()



In [ ]:
# # How do we know the ML didnt just 'get lucky'?
# #  well, luck wouldnt usually produce asymmetric errors. But...

# # We can look at label shuffling (the gold standard):

# y_shuffled = np.random.permutation(y)
# X_train_shuffled, X_test_shuffled, y_train_shuffled, y_test_shuffled = train_test_split(
#     X_pca, y_shuffled, test_size=0.3
# )
# clf.fit(X_train_shuffled, y_train_shuffled)
# print(clf.score(X_test_shuffled, y_test_shuffled))

# # If this is ~0.5 then the classifier is not exploiting artifacts and is learning real structure. 


In [ ]:
# # It seems that PCA1 has the most information, likely to be the chirp mass (since the distance is constant across all BNSs and BBHs).

# #     PC1 alone explains ~50% of the variance
# #     PC1 + PC2 ≈ 60%
# #     By PC5 you’re already at ~70%

# # This means:
# #     The waveform manifold is very low-dimensional.
# # That is what we expect physically: inspiral waveforms are essentially parameterized by chirp mass + total mass when spins and eccentricity are ignored.

# # TEST: Remove PC1 and retrain -> Accuracy should drop significantly.


# clf.fit(X_pca[:,1:], y)

# y_pred = clf.predict(X_test)

# print(classification_report(
#     y_test, y_pred,
#     target_names=["BNS", "BBH"]
# ))



In [ ]:
MODE = "low_mass_train_high_mass_test"   # or "high_mass_train_low_mass_test"

# =============================================================
# Cross-population generalization (FIXED)
# =============================================================

MODE = "low_mass_train_high_mass_test"   # or "high_mass_train_low_mass_test"

def generate_dataset(bns_range, bbh_range, n_each=200):
    X_out, y_out = [], []

    # BNS
    for _ in range(n_each):
        m1 = np.random.uniform(*bns_range)
        m2 = np.random.uniform(*bns_range)

        hp, _ = get_td_waveform(
            approximant="TaylorF2",
            mass1=m1, mass2=m2,
            delta_t=delta_t,
            f_lower=f_lower
        )

        h = hp.numpy()[-window_size:]
        if len(h) < window_size:
            h = np.pad(h, (window_size - len(h), 0))

        X_out.append(h)
        y_out.append(0)

    # BBH
    for _ in range(n_each):
        m1 = np.random.uniform(*bbh_range)
        m2 = np.random.uniform(*bbh_range)

        hp, _ = get_td_waveform(
            approximant="IMRPhenomD",
            mass1=m1, mass2=m2,
            delta_t=delta_t,
            f_lower=f_lower
        )

        h = hp.numpy()[-window_size:]
        if len(h) < window_size:
            h = np.pad(h, (window_size - len(h), 0))

        X_out.append(h)
        y_out.append(1)

    return np.array(X_out), np.array(y_out)


if MODE == "low_mass_train_high_mass_test":
    X_train_raw, y_train = generate_dataset(
        bns_range=(1.2, 1.4),
        bbh_range=(8, 12)
    )
    X_test_raw, y_test = generate_dataset(
        bns_range=(1.4, 1.6),
        bbh_range=(20, 30)
    )

elif MODE == "high_mass_train_low_mass_test":
    X_train_raw, y_train = generate_dataset(
        bns_range=(1.4, 1.6),
        bbh_range=(20, 30)
    )
    X_test_raw, y_test = generate_dataset(
        bns_range=(1.2, 1.4),
        bbh_range=(8, 12)
    )

# --- Preprocessing (train → test, no leakage!)
X_train_log = np.log10(np.abs(X_train_raw) + 1e-22)
X_test_log  = np.log10(np.abs(X_test_raw)  + 1e-22)

X_train_scaled = scaler.fit_transform(X_train_log)
X_test_scaled  = scaler.transform(X_test_log)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

clf = LogisticRegression(max_iter=500)
clf.fit(X_train_pca, y_train)

y_pred = clf.predict(X_test_pca)

print(f"\nCross-population test: {MODE}")
print(classification_report(
    y_test, y_pred,
    target_names=["BNS", "BBH"]
))



#  Interpretation
# “ML classifiers are excellent interpolators, but poor extrapolators.”
    # In physics terms:
    # “Morphology learned from low-mass systems does not generalize to high-mass systems unless you include them in training.”
    # A perfect discussion point: shows the limitations of naive ML and the importance of representative datasets.


    # Recall of BNS = 1.0 → every test waveform labeled as BNS was classified as BNS.
    # Recall of BBH = 0.0 → every BBH test waveform was misclassified as BNS.
    # Overall accuracy = 0.5 → random guessing baseline for two-class problem.
    
# Demonstrates ML cannot discover new physics outside the training distribution — a key lesson
